In [1]:
import os, io, time, json, hashlib, pathlib, sys
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import re
import matplotlib.pyplot as plt
from urllib.parse import urlparse
from datetime import datetime, timedelta


from config import *
from functions2 import *

In [2]:
ticker = 'GBPCHF.FOREX'
START = '2020-01-01'
MAX_AGE = 24
print(f'START {START}')
print(f'MAX AGE {MAX_AGE} hours')
params = {
    'from': START,  # EODHD uses from/to
    'to': today,
    'api_token': EOD_API
}
url = (f'https://eodhd.com/api/eod/{ticker}')


START 2020-01-01
MAX AGE 24 hours


In [3]:
df = fetch_csv_robust(url, params=params, ticker=ticker, max_age=MAX_AGE)
s = sort_cols(df)


GBPCHF.FOREX - downloading fresh data
saving  cache/GBPCHF.FOREX.csv


In [9]:
plt.style.use('dark_background')
n_df = pd.DataFrame([10,9,8,4,2,5,6,1])
n_df = n_df.rename(columns={0: 'prices'})
# print(f'n_df\n{n_df.head(2)}')
n_df.index = pd.to_datetime([
'2025-08-14' ,
'2025-08-15' ,
'2025-08-18' ,
'2025-08-19' ,
'2025-08-20' ,
'2025-08-21' ,
'2025-08-22' ,
'2025-08-25' ,
])
days = 2
n_df['span'] = (n_df.index - n_df.index.to_series().shift(2)).dt.days

# n_df = n_df[n_df['span'] == 2] # ?????
# print(f'n_df filtered\n{n_df['prices']}')

n_df['logr'] = np.log(n_df['prices']).diff()
# print(f'n_s\n{n_s}')
# print(f'rn_s\n{rn_s}')

n_df['ndayrets'] = n_df['logr'].rolling(days, min_periods=days).sum()
n_df['cost'] = n_df['span'] * 0.01
n_df['pass'] = n_df['ndayrets'] < n_df['cost']

print(f'n_df\n{n_df}')

# a = n_s.iloc[-4]
# b = n_s.iloc[-1]
# print('a,b', a,b)
# drop = a -b
# print(drop)
# print(s.iloc[-1])
# drop = s.iloc[-1] / s.iloc[-5]
# print(drop)
#  define carry, buffer
# plt.plot(n_s)
# plt.plot(n_s.index, n_s, a )
# plt.scatter(n_s.index[-4], n_s.iloc[-4], color='red')

n_df
            prices  span      logr  ndayrets  cost   pass
2025-08-14      10   NaN       NaN       NaN   NaN  False
2025-08-15       9   NaN -0.105361       NaN   NaN  False
2025-08-18       8   4.0 -0.117783 -0.223144  0.04   True
2025-08-19       4   4.0 -0.693147 -0.810930  0.04   True
2025-08-20       2   2.0 -0.693147 -1.386294  0.02   True
2025-08-21       5   2.0  0.916291  0.223144  0.02  False
2025-08-22       6   2.0  0.182322  1.098612  0.02  False
2025-08-25       1   4.0 -1.791759 -1.609438  0.04   True
